In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [16]:
# DATA_PATH = "/kaggle/input/bot-detection-data/bot_detection_data.csv"
DATA_PATH = "/kaggle/input/bot-detection-data/training_data.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)

(1562, 20)


In [17]:
df.head()

,id,id_str,screen_name,location,description,url,followers_count,friends_count,listedcount,created_at,favourites_count,verified,statuses_count,lang,status,default_profile,default_profile_image,has_extended_profile,name,bot
0,1.953701e+08,195370058,kanyejordan,NaN,This is what I do. I drop truth bombs.,NaN,2925,3,139,9/26/2010 14:45,0,False,708,en,"Status(in_reply_to_status_id=None, favorited=F...",True,False,False,Kanye Jordan,1
1,7.950000e+17,7.95E+17,astronaut_bot,NaN,Keeping an eye on astronauts coming and going....,NaN,9,0,5,Fri Nov 04 12:11:27 +0000 2016,0,False,6,en,{'created_at': 'Tue Nov 22 16:52:31 +0000 2016...,True,False,False,Astronaut Notifier,1
2,2.976541e+09,2976541239,TheRiddlerBot,"Coimbra, Portugal",Solve the riddle by replying only the name of ...,https://t.co/1v8BON9QpT,132,46,24,1/13/2015 15:10,740,False,7346,en,"Status(contributors=None, truncated=False, tex...",True,False,False,TheRiddlerBot,1
3,2.243832e+08,224383150,mlegoudes262,NaN,NaN,NaN,54,1351,0,Wed Dec 08 21:29:31 +0000 2010,2,False,6,en,"{'truncated': False, 'entities': {'user_mentio...",True,False,False,Laurie Poulsen,1
4,1.134712e+07,11347122,GavinNewsom,California,Husband & father. 49th Lt. Gov. of California ...,https://t.co/XrGnfzTDJD,1300380,24248,7089,Wed Dec 19 19:53:42 +0000 2007,4184,True,8536,en,"{u'contributors': None, u'truncated': True, u'...",False,False,False,Gavin Newsom,0


In [18]:
FEATURES = [
    "followers_count",
    "friends_count",
    "listedcount",
    "favourites_count",
    "statuses_count",
    "verified",
    "default_profile",
    "default_profile_image",
    "has_extended_profile"
]

X = df[FEATURES].fillna(0)
y = df["bot"]

In [19]:
bool_cols = [
    "verified",
    "default_profile",
    "default_profile_image",
    "has_extended_profile"
]

for col in bool_cols:
    X[col] = X[col].astype(int)

In [20]:
X["follow_ratio"] = X["followers_count"] / (X["friends_count"] + 1)

In [21]:
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

X["account_age_days"] = (
    pd.Timestamp.now() - df["created_at"]
).dt.days.fillna(0)


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [23]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)


RandomForestClassifier(class_weight='balanced', max_depth=20,
                       min_samples_leaf=2, n_estimators=300, n_jobs=-1,
                       random_state=42)

In [24]:
preds = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))


Accuracy: 0.8785942492012779
              precision    recall  f1-score   support

           0       0.90      0.87      0.89       169
           1       0.85      0.89      0.87       144

    accuracy                           0.88       313
   macro avg       0.88      0.88      0.88       313
weighted avg       0.88      0.88      0.88       313



In [25]:
imp = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

print(imp)

                  feature  importance
1           friends_count    0.204309
9            follow_ratio    0.144836
3        favourites_count    0.135528
0         followers_count    0.109556
5                verified    0.099516
10       account_age_days    0.090862
2             listedcount    0.088300
4          statuses_count    0.076216
6         default_profile    0.039780
8    has_extended_profile    0.008163
7   default_profile_image    0.002935


In [26]:
import joblib

joblib.dump(rf, "bot_model.joblib")

['bot_model.joblib']

In [27]:
# ✅ After training RF
print("RF trained feature count:", len(rf.feature_names_in_))
print("RF trained feature names:")
print(list(rf.feature_names_in_))


RF trained feature count: 11
RF trained feature names:
['followers_count', 'friends_count', 'listedcount', 'favourites_count', 'statuses_count', 'verified', 'default_profile', 'default_profile_image', 'has_extended_profile', 'follow_ratio', 'account_age_days']
